# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [5]:
# Part 1 - useful imports
from pyspark.sql.functions import (
    col, avg, hour, to_date, date_format,
    monotonically_increasing_id, unix_timestamp,
    row_number
)
from pyspark.sql.window import Window

In [6]:
# Add a unique key for each trip
df_trips = df_trips.withColumn(
    "trip_id",
    monotonically_increasing_id()
)

df_trips.select("trip_id").show(5)

+----------+
|   trip_id|
+----------+
|8589934592|
|8589934593|
|8589934594|
|8589934595|
|8589934596|
+----------+
only showing top 5 rows


In [7]:
# Add useful columns for the next questions
df_trips = (
    df_trips
    .withColumn(
        "duration_minutes",
        (
            unix_timestamp("tpep_dropoff_datetime")
            - unix_timestamp("tpep_pickup_datetime")
        ) / 60
    )
    .withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", date_format("tpep_pickup_datetime", "EEEE"))
)

In [8]:
# Which trip has the highest passenger count?
df_trips.orderBy(
    col("passenger_count").desc()
).select(
    "trip_id",
    "passenger_count",
    "trip_distance",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).show(1, truncate=False)

+----------+---------------+-------------+--------------------+---------------------+
|trip_id   |passenger_count|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+----------+---------------+-------------+--------------------+---------------------+
|8590884548|9.0            |0.0          |2019-01-05 13:12:29 |2019-01-05 13:12:32  |
+----------+---------------+-------------+--------------------+---------------------+
only showing top 1 row


In [9]:
# What is the average passenger count?
df_trips.select(
    avg("passenger_count").alias("average_passenger_count")
).show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



In [10]:
# Shortest / longest trip by distance and by time
print("Shortest trip by distance")
df_trips.orderBy(col("trip_distance").asc()).select(
    "trip_id", "trip_distance", "duration_minutes"
).show(1)

print("Longest trip by distance")
df_trips.orderBy(col("trip_distance").desc()).select(
    "trip_id", "trip_distance", "duration_minutes"
).show(1)

print("Shortest trip by time")
df_trips.orderBy(col("duration_minutes").asc()).select(
    "trip_id", "trip_distance", "duration_minutes"
).show(1)

print("Longest trip by time")
df_trips.orderBy(col("duration_minutes").desc()).select(
    "trip_id", "trip_distance", "duration_minutes"
).show(1)

Shortest trip by distance
+----------+-------------+-----------------+
|   trip_id|trip_distance| duration_minutes|
+----------+-------------+-----------------+
|8589934594|          0.0|4.166666666666667|
+----------+-------------+-----------------+
only showing top 1 row
Longest trip by distance
+----------+-------------+-----------------+
|   trip_id|trip_distance| duration_minutes|
+----------+-------------+-----------------+
|8596008683|        831.8|9.483333333333333|
+----------+-------------+-----------------+
only showing top 1 row
Shortest trip by time
+----------+-------------+----------------+
|   trip_id|trip_distance|duration_minutes|
+----------+-------------+----------------+
|8591137776|          3.3|        -84280.5|
+----------+-------------+----------------+
only showing top 1 row
Longest trip by time
+----------+-------------+-----------------+
|   trip_id|trip_distance| duration_minutes|
+----------+-------------+-----------------+
|8590002859|          1.2|43648.

In [11]:
# Busiest / slowest single day
daily_trips = df_trips.groupBy("pickup_date").count()

print("Busiest day")
daily_trips.orderBy(col("count").desc()).show(1)

print("Slowest day")
daily_trips.orderBy(col("count").asc()).show(1)

Busiest day
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
+-----------+------+
only showing top 1 row
Slowest day
+-----------+-----+
|pickup_date|count|
+-----------+-----+
| 2019-05-20|    1|
+-----------+-----+
only showing top 1 row


In [12]:
# Busiest / slowest time of day (hour)
hourly_trips = df_trips.groupBy("pickup_hour").count()

print("Busiest hour")
hourly_trips.orderBy(col("count").desc()).show(1)

print("Slowest hour")
hourly_trips.orderBy(col("count").asc()).show(1)

hourly_trips.orderBy("pickup_hour").show(24)

Busiest hour
+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|515390|
+-----------+------+
only showing top 1 row
Slowest hour
+-----------+-----+
|pickup_hour|count|
+-----------+-----+
|          4|61424|
+-----------+-----+
only showing top 1 row
+-----------+------+
|pickup_hour| count|
+-----------+------+
|          0|207842|
|          1|149254|
|          2|109421|
|          3| 78086|
|          4| 61424|
|          5| 75533|
|          6|178598|
|          7|304858|
|          8|373742|
|          9|365935|
|         10|361390|
|         11|375441|
|         12|401173|
|         13|404153|
|         14|433139|
|         15|452691|
|         16|420843|
|         17|468479|
|         18|515390|
|         19|475186|
|         20|423156|
|         21|409901|
|         22|369041|
|         23|281941|
+-----------+------+



In [13]:
# On average, which day of the week is busiest / slowest?
# First count trips per calendar date, then average those counts by weekday.
daily_weekday = (
    df_trips
    .groupBy("pickup_date", "day_of_week")
    .count()
)

weekday_average = (
    daily_weekday
    .groupBy("day_of_week")
    .agg(avg("count").alias("average_trips"))
)

print("Average trips by weekday")
weekday_average.orderBy(col("average_trips").desc()).show()

print("Busiest weekday on average")
weekday_average.orderBy(col("average_trips").desc()).show(1)

print("Slowest weekday on average")
weekday_average.orderBy(col("average_trips").asc()).show(1)

Average trips by weekday
+-----------+------------------+
|day_of_week|     average_trips|
+-----------+------------------+
|   Thursday| 193863.2857142857|
|     Friday|155316.42857142858|
|   Saturday|144283.57142857142|
|  Wednesday|140584.88888888888|
|    Tuesday|          120908.4|
|     Sunday|        107488.125|
|     Monday|           90812.1|
+-----------+------------------+

Busiest weekday on average
+-----------+-----------------+
|day_of_week|    average_trips|
+-----------+-----------------+
|   Thursday|193863.2857142857|
+-----------+-----------------+
only showing top 1 row
Slowest weekday on average
+-----------+-------------+
|day_of_week|average_trips|
+-----------+-------------+
|     Monday|      90812.1|
+-----------+-------------+
only showing top 1 row


In [14]:
# Does trip distance or number of passengers affect tip amount?
corr_distance_tip = df_trips.stat.corr(
    "trip_distance",
    "tip_amount"
)

corr_passenger_tip = df_trips.stat.corr(
    "passenger_count",
    "tip_amount"
)

print("Correlation trip_distance / tip_amount:", corr_distance_tip)
print("Correlation passenger_count / tip_amount:", corr_passenger_tip)

Correlation trip_distance / tip_amount: 0.5269200663652668
Correlation passenger_count / tip_amount: 0.004431051585116288


**Interpretation:** correlation measures the strength of a linear relationship between two variables.  
A positive value means they tend to increase together, a negative value means one tends to decrease when the other increases, and a value close to 0 means little linear relationship. Correlation does not prove causation.

In [15]:
# Highest "extra" charge and associated trip
df_trips.orderBy(
    col("extra").desc()
).select(
    "trip_id",
    "extra",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).show(1, truncate=False)

+----------+------+-----------+----------+------------+--------------------+---------------------+
|trip_id   |extra |fare_amount|tip_amount|total_amount|tpep_pickup_datetime|tpep_dropoff_datetime|
+----------+------+-----------+----------+------------+--------------------+---------------------+
|8595258075|535.38|355676.98  |0.0       |356214.78   |2019-01-23 08:58:09 |2019-01-23 08:58:09  |
+----------+------+-----------+----------+------------+--------------------+---------------------+
only showing top 1 row


In [16]:
# Look for strange values / possible outliers
df_trips.select(
    "passenger_count",
    "trip_distance",
    "duration_minutes",
    "fare_amount",
    "tip_amount",
    "extra",
    "total_amount"
).describe().show()

df_trips.filter(
    (col("trip_distance") <= 0)
    | (col("duration_minutes") <= 0)
    | (col("fare_amount") < 0)
    | (col("tip_amount") < 0)
).select(
    "trip_id",
    "trip_distance",
    "duration_minutes",
    "fare_amount",
    "tip_amount"
).show(20, truncate=False)

+-------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+
|summary|   passenger_count|     trip_distance|  duration_minutes|      fare_amount|        tip_amount|             extra|     total_amount|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+
|  count|           7667945|           7696617|           7696617|          7696617|           7696617|           7696617|          7696617|
|   mean|1.5670317144945614|2.8301461681153532|16.551081570422276|12.52967677747685|1.8208300763883147|0.3374054146126797|15.81065134371489|
| stddev|1.2244198591042095| 3.774548394256295| 81.67539611217202|261.5897471783846|2.4994631914320986|0.5313564053935059|261.8117056584905|
|    min|               0.0|               0.0|          -84280.5|           -362.0|             -63.5|             -60.0|           -362.8|
|    max|    

### Outlier reasoning

Some records can be considered suspicious and should be checked:

- A negative trip distance is not physically meaningful.
- A negative trip duration means the drop-off time is before the pickup time.
- A negative fare or tip can be a correction/refund, but it may also indicate a data-quality problem.
- A zero distance or zero duration can happen in special cases, so it should be investigated before removing the record.

For this exploratory analysis, I identify these values as possible outliers rather than deleting them automatically.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [17]:
# Load the Taxi Zone Lookup table
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_file = "taxi_zone_lookup.csv"

response = requests.get(zone_url)

if response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(response.content)

df_zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zone_file)
)

df_zones.show(5, truncate=False)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


In [18]:
# Join pickup and dropoff location IDs with borough / zone names
pickup_zones = df_zones.select(
    col("LocationID").alias("PU_ID"),
    col("Borough").alias("pickup_borough"),
    col("Zone").alias("pickup_zone")
)

dropoff_zones = df_zones.select(
    col("LocationID").alias("DO_ID"),
    col("Borough").alias("dropoff_borough"),
    col("Zone").alias("dropoff_zone")
)

df_joined = (
    df_trips
    .join(
        pickup_zones,
        df_trips.PULocationID == pickup_zones.PU_ID,
        "left"
    )
    .drop("PU_ID")
    .join(
        dropoff_zones,
        df_trips.DOLocationID == dropoff_zones.DO_ID,
        "left"
    )
    .drop("DO_ID")
)

In [19]:
# Which borough had the most pickups? dropoffs?
print("Pickups by borough")
df_joined.groupBy("pickup_borough") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

print("Dropoffs by borough")
df_joined.groupBy("dropoff_borough") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

Pickups by borough
+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|6950965|
|        Queens| 471173|
|       Unknown| 159815|
|      Brooklyn|  91905|
|         Bronx|  18062|
|           N/A|   3890|
|           EWR|    446|
| Staten Island|    361|
+--------------+-------+

Dropoffs by borough
+---------------+-------+
|dropoff_borough|  count|
+---------------+-------+
|      Manhattan|6817355|
|         Queens| 340972|
|       Brooklyn| 301105|
|        Unknown| 149097|
|          Bronx|  58085|
|            N/A|  16904|
|            EWR|  10914|
|  Staten Island|   2185|
+---------------+-------+



In [20]:
# Busy / slow times by borough
borough_hour = (
    df_joined
    .filter(col("pickup_borough").isNotNull())
    .groupBy("pickup_borough", "pickup_hour")
    .count()
)

busy_window = Window.partitionBy(
    "pickup_borough"
).orderBy(
    col("count").desc()
)

slow_window = Window.partitionBy(
    "pickup_borough"
).orderBy(
    col("count").asc()
)

print("Busiest hour by borough")
borough_hour.withColumn(
    "rank", row_number().over(busy_window)
).filter(
    col("rank") == 1
).drop("rank").show()

print("Slowest hour by borough")
borough_hour.withColumn(
    "rank", row_number().over(slow_window)
).filter(
    col("rank") == 1
).drop("rank").show()

Busiest hour by borough
+--------------+-----------+------+
|pickup_borough|pickup_hour| count|
+--------------+-----------+------+
|         Bronx|          7|  1803|
|      Brooklyn|          8|  6935|
|           EWR|         15|    54|
|     Manhattan|         18|471539|
|           N/A|         19|   214|
|        Queens|         16| 29885|
| Staten Island|          8|    36|
|       Unknown|         18| 10751|
+--------------+-----------+------+

Slowest hour by borough
+--------------+-----------+-----+
|pickup_borough|pickup_hour|count|
+--------------+-----------+-----+
|         Bronx|          3|  225|
|      Brooklyn|          3| 1919|
|           EWR|         23|    1|
|     Manhattan|          4|53447|
|           N/A|          6|   88|
|        Queens|          3| 3085|
| Staten Island|          1|    3|
|       Unknown|          4| 1465|
+--------------+-----------+-----+



In [21]:
# Busiest day of the week by borough
borough_weekday = (
    df_joined
    .filter(col("pickup_borough").isNotNull())
    .groupBy("pickup_borough", "day_of_week")
    .count()
)

weekday_window = Window.partitionBy(
    "pickup_borough"
).orderBy(
    col("count").desc()
)

borough_weekday.withColumn(
    "rank", row_number().over(weekday_window)
).filter(
    col("rank") == 1
).drop("rank").show()

+--------------+-----------+-------+
|pickup_borough|day_of_week|  count|
+--------------+-----------+-------+
|         Bronx|   Thursday|   3121|
|      Brooklyn|    Tuesday|  15779|
|           EWR|  Wednesday|     83|
|     Manhattan|   Thursday|1229554|
|           N/A|    Tuesday|    703|
|        Queens|   Thursday|  78972|
| Staten Island|     Friday|     64|
|       Unknown|   Thursday|  28929|
+--------------+-----------+-------+



In [22]:
# Average trip distance by borough
df_joined.groupBy(
    "pickup_borough"
).agg(
    avg("trip_distance").alias("average_trip_distance")
).orderBy(
    col("average_trip_distance").desc()
).show()

+--------------+---------------------+
|pickup_borough|average_trip_distance|
+--------------+---------------------+
| Staten Island|   12.503601108033246|
|        Queens|   11.283218499361993|
|         Bronx|    7.233194552098303|
|      Brooklyn|    4.787677275447492|
|           N/A|    3.193850899742941|
|           EWR|    2.641098654708519|
|       Unknown|    2.415464130400774|
|     Manhattan|   2.2286693358402596|
+--------------+---------------------+



In [23]:
# Average trip fare by borough
df_joined.groupBy(
    "pickup_borough"
).agg(
    avg("fare_amount").alias("average_fare")
).orderBy(
    col("average_fare").desc()
).show()

+--------------+------------------+
|pickup_borough|      average_fare|
+--------------+------------------+
|           EWR| 76.24024663677126|
|           N/A|  59.5731593830335|
| Staten Island|45.289861495844896|
|        Queens| 35.14462651722029|
|         Bronx| 26.26890543682963|
|      Brooklyn|18.649132800172286|
|       Unknown|14.944423051653523|
|     Manhattan|10.792468572351568|
+--------------+------------------+



In [24]:
# Highest / lowest fare and associated borough
print("Highest fare")
df_joined.orderBy(
    col("fare_amount").desc()
).select(
    "trip_id",
    "fare_amount",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone"
).show(1, truncate=False)

print("Lowest fare")
df_joined.orderBy(
    col("fare_amount").asc()
).select(
    "trip_id",
    "fare_amount",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone"
).show(1, truncate=False)

Highest fare
+----------+-----------+--------------+---------------------+---------------+------------+
|trip_id   |fare_amount|pickup_borough|pickup_zone          |dropoff_borough|dropoff_zone|
+----------+-----------+--------------+---------------------+---------------+------------+
|8592434247|623259.86  |Manhattan     |Upper East Side South|Manhattan      |Flatiron    |
+----------+-----------+--------------+---------------------+---------------+------------+
only showing top 1 row
Lowest fare
+----------+-----------+--------------+-----------+---------------+------------+
|trip_id   |fare_amount|pickup_borough|pickup_zone|dropoff_borough|dropoff_zone|
+----------+-----------+--------------+-----------+---------------+------------+
|8594825241|-362.0     |Queens        |JFK Airport|Queens         |JFK Airport |
+----------+-----------+--------------+-----------+---------------+------------+
only showing top 1 row


In [25]:
# Load January 2025 taxi data
url_2025 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
file_2025 = "yellow_tripdata_2025-01.parquet"

response = requests.get(url_2025)

if response.status_code == 200:
    with open(file_2025, "wb") as f:
        f.write(response.content)

df_2025 = spark.read.parquet(file_2025)

In [26]:
# Compare average metrics: January 2019 vs January 2025
print("Average metrics - January 2019")
df_trips.select(
    avg("passenger_count").alias("avg_passengers"),
    avg("trip_distance").alias("avg_distance"),
    avg("fare_amount").alias("avg_fare"),
    avg("tip_amount").alias("avg_tip"),
    avg("total_amount").alias("avg_total")
).show()

print("Average metrics - January 2025")
df_2025.select(
    avg("passenger_count").alias("avg_passengers"),
    avg("trip_distance").alias("avg_distance"),
    avg("fare_amount").alias("avg_fare"),
    avg("tip_amount").alias("avg_tip"),
    avg("total_amount").alias("avg_total")
).show()

Average metrics - January 2019
+------------------+------------------+-----------------+------------------+-----------------+
|    avg_passengers|      avg_distance|         avg_fare|           avg_tip|        avg_total|
+------------------+------------------+-----------------+------------------+-----------------+
|1.5670317144945614|2.8301461681153532|12.52967677747685|1.8208300763883147|15.81065134371489|
+------------------+------------------+-----------------+------------------+-----------------+

Average metrics - January 2025
+------------------+-----------------+-----------------+------------------+------------------+
|    avg_passengers|     avg_distance|         avg_fare|           avg_tip|         avg_total|
+------------------+-----------------+-----------------+------------------+------------------+
|1.2978589658806226|5.855126178843539|17.08180276045484|2.9598127862758044|25.611291697280986|
+------------------+-----------------+-----------------+------------------+-------

**2019 vs 2025:** after running the cell above, compare the two result tables and note which average values increased or decreased. This is a descriptive comparison of the two January datasets.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [27]:
# Create temporary views for Spark SQL
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [28]:
# SQL question 1 - Average passenger count
spark.sql("""
SELECT
    AVG(passenger_count) AS average_passenger_count
FROM trips
""").show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



In [29]:
# SQL question 2 - Busiest hour
spark.sql("""
SELECT
    HOUR(tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS number_of_trips
FROM trips
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY number_of_trips DESC
LIMIT 1
""").show()

+-----------+---------------+
|pickup_hour|number_of_trips|
+-----------+---------------+
|         18|         515390|
+-----------+---------------+



In [30]:
# SQL question 3 - Borough with the most pickups
# This question contains the required JOIN.
spark.sql("""
SELECT
    z.Borough AS pickup_borough,
    COUNT(*) AS number_of_pickups
FROM trips t
LEFT JOIN zones z
    ON t.PULocationID = z.LocationID
GROUP BY z.Borough
ORDER BY number_of_pickups DESC
""").show()

+--------------+-----------------+
|pickup_borough|number_of_pickups|
+--------------+-----------------+
|     Manhattan|          6950965|
|        Queens|           471173|
|       Unknown|           159815|
|      Brooklyn|            91905|
|         Bronx|            18062|
|           N/A|             3890|
|           EWR|              446|
| Staten Island|              361|
+--------------+-----------------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing